In [63]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

df = pd.read_csv("roadmap_tracker.csv", encoding="utf-8-sig")
df["repeated"] = pd.to_numeric(df["repeated"], errors="coerce").fillna(0)    # повторения → числа

S = 0.7                                          # масштаб: 0.7 = минус 30%; 1 — исходный размер
CELL, LABEL_H, HEAD_H = 24 * S, 170 * S, 28 * S  # кубик, полоса под подписи разделов, полоса под название этапа
order = ["eda", "preprocessing", "model", "activation", "loss", "optimizer", "metrics"]

cols_z, cols_text, chapters, heads = [], [], [], []
for step in order:
    start = len(cols_z)                                                      # первый столбец этапа
    for chapter, grp in df[df["step"] == step].groupby("chapter", sort=False):
        chapters.append((len(cols_z), chapter.split("/")[-1]))               # столбец раздела и его подпись
        cols_z.append(list(grp["repeated"].clip(upper=4)))                   # кубики = темы раздела
        cols_text.append([f"<b>{t}</b><br>{c}<br>повторений: {int(n)}"
                          for t, c, n in zip(grp["topic"], grp["chapter"], grp["repeated"])])
    heads.append(((start + len(cols_z) - 1) / 2, step))                      # центр этапа
    cols_z.append([]); cols_text.append([])                                  # отступ между этапами
cols_z.pop(); cols_text.pop()

height = max(len(c) for c in cols_z)
z = np.array([c + [np.nan] * (height - len(c)) for c in cols_z], dtype=float).T         # .T — поворот
text = np.array([c + [""] * (height - len(c)) for c in cols_text], dtype=object).T

fig = go.Figure(go.Heatmap(
    z=z, text=text, hoverinfo="text", xgap=4 * S, ygap=4 * S,
    colorscale=["#ebedf0", "#9be9a8", "#40c463", "#30a14e", "#216e39"],
    zmin=0, zmax=4,
    colorbar=dict(tickvals=[0, 1, 2, 3, 4], ticktext=["0", "1", "2", "3", "4+"],
                  thickness=12 * S, len=1, y=0.5, yanchor="middle", outlinewidth=0,
                  tickfont=dict(size=11 * S)),
))

for x, step in heads:                                                        # названия этапов — по центру своих столбцов
    fig.add_annotation(x=x, y=-0.7, xanchor="center", yanchor="bottom",
                       text=f"<b>{step.upper()}</b>",
                       showarrow=False, font=dict(size=13 * S, color="#24292f"))

for x, name in chapters:                                                     # подписи разделов — вертикально, над этапом
    fig.add_annotation(x=x, y=-0.7, yshift=HEAD_H, text=name, textangle=-90,
                       xanchor="center", yanchor="bottom",
                       showarrow=False, font=dict(size=11 * S, color="#57606a"))

fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False, autorange="reversed", scaleanchor="x")
fig.update_layout(
    title=dict(text=f"<b>ROADMAP-ML</b> · {int((df['repeated'] > 0).sum())} из {len(df)} тем повторено",
               x=0.01, y=0.98, yanchor="top", font=dict(size=20 * S)),
    font=dict(family="Inter, Helvetica, Arial"),
    width=CELL * z.shape[1] + 140 * S, height=CELL * height + LABEL_H + HEAD_H + 110 * S,
    plot_bgcolor="white", paper_bgcolor="white",
    margin=dict(t=LABEL_H + HEAD_H + 70 * S, l=10, r=10, b=10),
)
fig.show()